# CRO — datos necesarios y análisis

Los archivos originales no contienen el historial de etapas, las evaluaciones de prospectos ni las interacciones necesarios para responder las preguntas del CRO.

**Para cubrir esa información faltante, creamos datos sintéticos.** Con ellos construimos las métricas y demostramos el análisis; no comprobamos qué ocurrió realmente en la empresa.

**Recorrido:** tablas a crear → datos a generar → validación → preguntas, métricas y gráficas.


In [1]:
import csv, hashlib, json, sqlite3, sys
from contextlib import closing
from pathlib import Path
from IPython.display import HTML, Markdown, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/cro_demo.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts.cro_demo import FIELDS, DICTIONARY, CSV_NAMES, model_html, preview_table
from scripts.cro_complete import validate_full
ESCENARIO = 'referencia'
assert ESCENARIO in ('referencia', 'conversion', 'demora')
BASE = ROOT / 'resultados/cro'
CARPETA = ROOT / 'datos/sinteticos/cro' / ESCENARIO
DB_CARPETA = ROOT / 'datos/bases/cro' / ESCENARIO
METRICAS = BASE / 'metricas' / ESCENARIO
manifest = json.loads((BASE / 'manifest.json').read_text())
display(Markdown(f'**Escenario:** {ESCENARIO}. Puedes cambiarlo arriba; empezamos con referencia.'))


**Escenario:** referencia. Puedes cambiarlo arriba; empezamos con referencia.

## 1. Tablas y datos que creamos

### Tablas a crear

- **Cuentas / prospectos:** empresas que entran y sus características.
- **Registros de etapas:** entradas, salidas y resultados en cada etapa.
- **Interacciones:** intentos de contacto y sus resultados.
- **Pagos:** eventos de pago y su validez.

**Etapas** es el catálogo auxiliar de nombres y orden del funnel. **Ganados** se calcula desde los registros de etapas; no es otra tabla de origen.

### Datos a generar

- **Cuentas:** identificador, fecha y tipo de entrada, canal, campaña, industria, producto, región, perfil y evaluación de calidad.
- **Registros de etapas:** cuenta, etapa, fechas de entrada y salida, destino, estado, motivo de pérdida y asignación de responsable.
- **Interacciones:** cuenta y visita de etapa, fecha, tipo de contacto, resultado y responsable.
- **Pagos:** identificador del pago, cuenta, fecha y estado; incluir historia previa para identificar nuevos pagadores.

Las tablas se conectan por `id_cuenta`; las interacciones también usan `id_registro` para identificar la visita de etapa. En esta demo, una cuenta representa una empresa con una oportunidad comercial.

A continuación revisamos qué representa cada fila y una muestra de los datos creados.


In [2]:
display(HTML('<style>.scroll{overflow-x:auto}.scroll table{border-collapse:collapse}.scroll td,.scroll th{padding:8px;border-bottom:1px solid #ddd;text-align:left;white-space:nowrap}details{margin:12px 0}summary{cursor:pointer}</style>'))
def leer_tabla(nombre):
    with (CARPETA / f'{CSV_NAMES.get(nombre,nombre)}.csv').open(encoding='utf-8', newline='') as f:
        reader=csv.DictReader(f)
        assert reader.fieldnames == FIELDS[nombre]
        return list(reader)
def mostrar_tabla(nombre,columnas):
    filas=leer_tabla(nombre)
    display(Markdown(f'**{len(filas):,} filas.** Muestra de 3 filas con las columnas principales.'))
    display(HTML(preview_table(columnas,[{k:v or None for k,v in r.items()} for r in filas[:3]])))
    definiciones=[dict(columna=k,significado=v[1]) for k,v in DICTIONARY[nombre][2].items()]
    detalle=preview_table(['columna','significado'],definiciones)
    archivo=CARPETA / f'{CSV_NAMES.get(nombre,nombre)}.csv'
    display(HTML('<details><summary>Ver significado de todas las columnas</summary>'+detalle+'</details>'))
    display(Markdown(f'Archivo completo: `{archivo}`'))
    return filas


### A. Cuentas: quiénes entraron

Una fila representa una **empresa potencial**. Por ejemplo: llegó por Marketing, pertenece a Comercio y entró por New. Nos permite contar entradas y comparar grupos similares.

También guarda la evaluación de calidad: si encaja en el perfil objetivo, tiene una necesidad cubierta e intención de compra. **Sin evaluación no significa mala calidad.** Los criterios ilustrativos están en el detalle de columnas y en las reglas del modelo.


In [3]:
cuentas=mostrar_tabla('dim_cuentas',['id_cuenta','fecha_captacion','canal','industria','tipo_entrada','calificado'])


**1,800 filas.** Muestra de 3 filas con las columnas principales.

id_cuenta,fecha_captacion,canal,industria,tipo_entrada,calificado
C01-00000,2026-01-04,Marketing pago,Servicios,New,0
C01-00001,2026-01-11,Orgánico,Comercio,New,—
C01-00002,2026-01-19,Referidos,Comercio,New,0


columna,significado
id_cuenta,Identificador de la cuenta
canal,Origen de captación
industria,Sector de actividad
tipo_entrada,"Entrada por New, SQL o compra desde producto"
fecha_captacion,Primera captación observada en la simulación
campana,Campaña de captación; no aplica a todos los canales
producto,Producto de interés
region,Región ilustrativa
perfil,Tamaño de empresa ilustrativo
responsable,Responsable inicial


Archivo completo: `./datos/sinteticos/cro/referencia/dim_cuentas.csv`

### B. Registros de etapas: qué pasó en cada etapa

Una fila representa **una visita de una empresa a una etapa**. Si entra a Working el día 10 y sale el 15, estuvo cinco días en esa visita. Fechas y resultado permiten calcular conversión, pérdidas y tiempos por etapa.

`avance` significa que pasó al destino indicado; puede saltar una etapa. `perdido` terminó sin avanzar. `abierto` sigue pendiente al corte. `ganado` llegó a Won. Una empresa puede tener varias filas y volver a una etapa: **filas no equivale a clientes**.


In [4]:
registros=mostrar_tabla('fact_etapas',['id_registro','id_cuenta','id_etapa','entrada','salida','estado','id_etapa_destino'])


**6,740 filas.** Muestra de 3 filas con las columnas principales.

id_registro,id_cuenta,id_etapa,entrada,salida,estado,id_etapa_destino
C01-00000-0,C01-00000,New,2026-01-04,2026-01-10,avance,Working
C01-00000-1,C01-00000,Working,2026-01-10,2026-01-13,avance,Engaged
C01-00000-2,C01-00000,Engaged,2026-01-13,2026-01-21,avance,SQL


columna,significado
id_registro,Identificador de esta entrada
id_cuenta,Cuenta que entró
id_etapa,Etapa a la que entró
entrada,Día de entrada a la etapa
salida,Día de avance o pérdida; vacío si abierto o Won
estado,"avance, perdido, abierto o ganado"
id_etapa_destino,Siguiente etapa alcanzada; solo en avances
motivo_perdida,"Motivo registrado de pérdida, no causa demostrada"
responsable_etapa,Responsable de la visita
fecha_asignacion_etapa,Asignación durante la visita


Archivo completo: `./datos/sinteticos/cro/referencia/fact_etapas.csv`

### C. Interacciones: qué atención recibió

Una fila representa **un intento de contacto**, como una llamada o un correo. `id_registro` indica en qué visita de etapa ocurrió.

Sirve para medir cuánto tardamos en intentar atender y cuántas visitas quedaron sin intento. `sin_respuesta` es un intento; `intercambio_efectivo` implica respuesta o conversación. **Esto mide atención; las fechas de etapas miden permanencia.**


In [5]:
interacciones=mostrar_tabla('interacciones',['id_cuenta','id_registro','fecha','tipo','resultado'])


**8,635 filas.** Muestra de 3 filas con las columnas principales.

id_cuenta,id_registro,fecha,tipo,resultado
C01-00000,C01-00000-0,2026-01-09,correo,sin_respuesta
C01-00000,C01-00000-1,2026-01-10,llamada,intercambio_efectivo
C01-00000,C01-00000-1,2026-01-11,llamada,intercambio_efectivo


columna,significado
id_interaccion,Identificador de interacción
id_cuenta,Cuenta atendida
id_registro,Visita de etapa atendida
fecha,Fecha del intento
tipo,Llamada o correo
resultado,Sin respuesta o intercambio efectivo
responsable,Responsable de la interacción


Archivo completo: `./datos/sinteticos/cro/referencia/interacciones.csv`

### D. Pagos: quién pagó y cuándo

Una fila representa **un evento de pago**. Solo `valido` cuenta. Una misma empresa puede tener pagos fallidos y varios pagos válidos.

Sirve para contar nuevos pagadores y compras desde producto. Un segundo pago no crea otro cliente nuevo. La demo incluye historia previa para distinguir quienes ya pagaban. **Won es un cierre comercial; el pago se comprueba aquí.**


In [6]:
pagos=mostrar_tabla('pagos',['id_pago','id_cuenta','fecha_pago','estado_pago'])


**2,440 filas.** Muestra de 3 filas con las columnas principales.

id_pago,id_cuenta,fecha_pago,estado_pago
C01-00000-fallido,C01-00000,2026-02-06,fallido
C01-00000-p1,C01-00000,2026-02-06,valido
C01-00000-p2,C01-00000,2026-03-08,valido


columna,significado
id_pago,Identificador de pago
id_cuenta,Cuenta pagadora
fecha_pago,Fecha del evento de pago
estado_pago,Valido o fallido; solo valido cuenta


Archivo completo: `./datos/sinteticos/cro/referencia/pagos.csv`

### Catálogo auxiliar: cuáles son las etapas

Una fila por nombre de etapa. `siguiente` indica la transición habitual que vamos a medir; Won no tiene siguiente.

**Cómo se conectan:** una cuenta tiene varias visitas y pagos; cada visita puede tener varias interacciones. No unimos todo directamente porque multiplicaría filas y distorsionaría las métricas.


In [7]:
etapas=leer_tabla('dim_etapas')
display(HTML(preview_table(FIELDS['dim_etapas'],[{k:v or None for k,v in r.items()} for r in etapas])))


id_etapa,orden,siguiente
New,1,Working
Working,2,Engaged
Engaged,3,SQL
SQL,4,Demo
Demo,5,Proposal
Proposal,6,Won
Won,7,—


## 2. Validación de los datos

Control técnico: comprueba integridad de archivos, relaciones y fechas. El detalle queda plegado.


In [8]:
for relative,digest in manifest['hashes'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==digest,relative
with closing(sqlite3.connect(DB_CARPETA/'modelo.sqlite')) as con:
    controles=validate_full(con,manifest['config']['cutoff'])
    for tabla in ('dim_cuentas','fact_etapas','interacciones','pagos','dim_etapas'):
        assert con.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]==len(leer_tabla(tabla))
display(HTML('<details><summary>Ver resultado de la validación</summary><p>Archivos íntegros; claves, relaciones, fechas e historia de pagos coherentes.</p></details>'))


## 3. Pregunta principal y recorrido del análisis

**¿La tasa de conversión de New a Won aumentó o disminuyó después del aumento de New?**

**Recorrido:** confirmar el aumento de New → comparar conversión → identificar canales afectados → revisar calidad → localizar etapas → investigar causas → proponer acciones.

**Población principal:** cuentas que entraron por New. Las entradas directas a SQL se evalúan desde SQL y por separado; entrar allí no demuestra por sí solo mayor calidad. Producto queda fuera de New → Won, pero se conserva como contexto del crecimiento total.

**Estado:** gráficas exploratorias disponibles. La respuesta consolidada antes/después y la validación de causas siguen pendientes.



### 3.1. ¿Cuánto creció New, frente a qué periodo y en qué segmentos?

**Qué medimos:** cuánto cambian las entradas a New frente a un periodo de referencia y en qué segmentos ocurre.

**Alcance mostrado ahora:** total general y canales, mes frente al anterior. Región y otros segmentos se revisarán si aportan diferencias relevantes; todavía no se ha completado toda la segmentación.

**Variación mensual:** `(New del mes − New del mes anterior) / New del mes anterior × 100`.

**Datos:** cuentas que entraron por New, fecha de entrada y atributos del segmento.


In [9]:
from scripts.cro_charts import grafica_new, CONFIG
figura_new=grafica_new(METRICAS/'demanda.csv', ESCENARIO)
figura_new.write_html(BASE/'graficas/01_new_tendencia.html',include_plotlyjs=True,config=CONFIG)
figura_new.show(renderer='plotly_mimetype',config=CONFIG)


In [10]:
def leer_metrica(nombre,dimension='total',**filtros):
    with (METRICAS/f'{nombre}.csv').open(encoding='utf-8',newline='') as f:
        return [r for r in csv.DictReader(f) if r['dimension']==dimension and all(r[k]==str(v) for k,v in filtros.items())]
demanda=leer_metrica('demanda')
columnas=['Mes','Entradas a New','Variación mensual (%)']
filas=[{'Mes':r['mes'],'Entradas a New':int(r['entradas_new']),
        'Variación mensual (%)':round(float(r['crecimiento_new_pct']),1) if r['crecimiento_new_pct'] else None}
       for r in demanda]
display(HTML('<details><summary>Ver valores de la gráfica</summary>'+preview_table(columnas,filas)+'</details>'))


Mes,Entradas a New,Variación mensual (%)
2026-01,127,—
2026-02,124,-2.4
2026-03,132,6.5
2026-04,133,0.8
2026-05,179,34.6
2026-06,183,2.2
2026-07,182,-0.5
2026-08,173,-4.9


#### Conclusión


In [11]:
# Genera la conclusión a partir de las métricas actuales.
from html import escape
from scripts.cro_insights import conclusion_new
lectura_new=conclusion_new(METRICAS/'demanda.csv')
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;">'+escape(lectura_new['texto'])+'</div>'))


#### Desglose por canal
Comparamos el aumento con los niveles anteriores y con lo que ocurrió después.


In [12]:
from scripts.cro_charts import grafica_new_por_canal
figura_canales=grafica_new_por_canal(METRICAS/'demanda.csv')
figura_canales.show(renderer='plotly_mimetype',config=CONFIG)


#### Conclusión por canal


In [13]:
from scripts.cro_insights import conclusion_new_por_canal
lectura_canales=conclusion_new_por_canal(METRICAS/'demanda.csv')
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;">'+escape(lectura_canales['texto']).replace('\n\n', '<br><br>')+'</div>'))


### 3.2. ¿Cómo entendemos la relación entre New y Won?

Primero medimos el tiempo observado de New a Won; después usamos ese plazo para comparar la conversión.


**Criterio de lectura:** comparar el periodo anterior al aumento con el posterior, globalmente y por canal. La tasa a 30 días mide conversión dentro de ese plazo; revisar 60/90 días antes de afirmar una pérdida final.


#### ¿Cuánto tiempo tardaron los New en llegar a Won?

**Qué medimos:** la mediana de días entre la primera entrada a New y Won.

**Alcance mostrado ahora:** 432 cuentas que entraron por New entre enero y agosto de 2026 y llegaron a Won hasta el 27 de septiembre. Mediana general y por mes de entrada.

**Métrica:** `mediana(fecha de Won − primera fecha de New)`, en días.

**Datos:** `datos/sinteticos/cro/referencia/ganados.csv`; campos `id_cuenta`, `tipo_entrada`, `fecha_entrada_inicial` y `fecha_won`. El código calcula la duración desde las fechas.

Esta medición incluye únicamente quienes llegaron a Won; no estima el tiempo de los casos pendientes ni la duración histórica previa al aumento.


In [ ]:
import pandas as pd
# Mediana desde las fechas de primera entrada a New y primer Won.
ganados_base=pd.read_csv(ROOT/'datos/sinteticos/cro/referencia/ganados.csv')
ganados_new=ganados_base.loc[ganados_base['tipo_entrada'].eq('New')].copy()
duracion=(pd.to_datetime(ganados_new['fecha_won'])-pd.to_datetime(ganados_new['fecha_entrada_inicial'])).dt.total_seconds()/86400
mediana_new_won=float(duracion.median())
mediana_texto=f'{mediana_new_won:g}'.replace('.',',')
# Validar que el fundamento registrado sigue coincidiendo con los datos.
assert mediana_new_won == 30, 'Revisar la justificación del plazo: cambió la mediana de referencia.'
assert duracion.notna().all() and duracion.ge(0).all()
ganados_new['mes_new']=pd.to_datetime(ganados_new['fecha_entrada_inicial']).dt.to_period('M').astype(str)
ganados_new['duracion_dias']=duracion
resumen_duracion=ganados_new.groupby('mes_new').agg(cuentas_ganadas=('id_cuenta','size'),mediana_dias=('duracion_dias','median')).reset_index()
resumen_duracion.columns=['Mes de entrada a New','Cuentas que llegaron a Won','Mediana (días)']
display(resumen_duracion)
display(HTML(f'<p style="font-size:19px;line-height:1.6"><strong>Resultado:</strong> la mediana de New a Won es {mediana_texto} días, calculada sobre {len(ganados_new)} cuentas ganadas. Este resultado fundamenta el plazo de comparación de 30 días.</p>'))


Mes de entrada a New,Cuentas que llegaron a Won,Mediana (días)
2026-01,38,29.5
2026-02,42,30.0
2026-03,50,29.0
2026-04,48,28.5
2026-05,61,30.0
2026-06,65,30.0
2026-07,58,30.0
2026-08,70,31.0


Medición de duración New → Won

In [ ]:
import plotly.graph_objects as go
figura_duracion=go.Figure(go.Scatter(x=resumen_duracion.iloc[:,0], y=resumen_duracion.iloc[:,2], mode="lines+markers", line=dict(color="#2563eb",width=3), hovertemplate="Mes de entrada: %{x}<br>Mediana: %{y} días<extra></extra>"))
figura_duracion.update_layout(title="Tiempo de New a Won por mes de entrada · 2026",template="plotly_white",height=400,font=dict(family="Arial, sans-serif",size=14),margin=dict(l=65,r=30,t=65,b=60),xaxis_title="Mes de entrada a New",yaxis_title="Mediana (días)")
figura_duracion.add_hline(y=mediana_new_won,line_dash="dot",line_color="#64748b",annotation_text=f"Mediana general: {mediana_texto} días",annotation_position="bottom right")
figura_duracion.show(renderer="plotly_mimetype",config=CONFIG)


Mediana de duración por mes de entrada

#### ¿Qué porcentaje de New llegó a Won en 30 días?

**Qué medimos:** la conversión de los New de cada mes dentro del plazo fundamentado por la mediana.

**Alcance mostrado ahora:** total general y canales, enero a agosto de 2026, con 30 días completos de seguimiento.

**Métrica:** `New del mes que llegó a Won en 30 días / entradas a New del mes × 100`.

**Datos:** primera entrada a New y fecha de Won de la misma cuenta; métricas en `cohortes.csv`.


In [15]:
from scripts.cro_charts import grafica_new_won
figura_new_won=grafica_new_won(METRICAS/'cohortes.csv',ESCENARIO,ventana=30)
figura_new_won.show(renderer='plotly_mimetype',config=CONFIG)


#### Desglose por canal


In [16]:
from scripts.cro_charts import grafica_conversion_por_canal
figura_conversion_canales=grafica_conversion_por_canal(METRICAS/'cohortes.csv',ventana=30)
figura_conversion_canales.show(renderer='plotly_mimetype',config=CONFIG)


#### Análisis de la conversión


In [17]:
from scripts.cro_insights import conclusion_conversion
lectura_conversion=conclusion_conversion(METRICAS/'cohortes.csv',ventana=30,mes_aumento=lectura_new['evidencia']['mes'])
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;">'+escape(lectura_conversion['texto']).replace('\n\n','<br><br>')+'</div>'))


### 3.3. ¿Cambió la calidad de los leads o el criterio para calificarlos?

**Comprobación previa:** ¿existe una puntuación de calidad de los leads?

- **Si existe:** analizar su evolución con una escala comparable.
- **Si no existe:** avanzar con los datos disponibles y proponer implementar la medición.

**Para este ejercicio:** generamos puntuaciones aleatorias de 0 a 100. No representan una evaluación real ni explican la caída de conversión.

**Qué medimos:** si cambió la puntuación media de los leads que entraron por New.

**Alcance mostrado ahora:** enero a agosto de 2026, global y por canal; comparación abril–mayo y evolución posterior. Todos los New tienen puntuación en esta demo.

**Métrica:** `suma de puntuaciones / número de New con puntuación`.

**Datos:** `dim_cuentas.csv` (cuenta, fecha de captación, entrada New y canal) y `puntuaciones_leads.csv` (cuenta y puntuación). No diseñamos un sistema de calificación; usamos sus valores como entrada.


In [ ]:
from scripts.cro_quality import analyze_scores
analisis_puntuacion=analyze_scores(CARPETA/"dim_cuentas.csv",CARPETA/"puntuaciones_leads.csv",month_of_increase=lectura_new["evidencia"]["mes"])
for figura in analisis_puntuacion["figures"]:
    figura.show(renderer="plotly_mimetype",config=CONFIG)


Puntuación media de leads

Puntuación media de leads

#### Conclusión


In [ ]:
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;">'+escape(analisis_puntuacion["reading"])+'</div>'))


Conclusión de puntuación

### 3.4. ¿Cambió la conversión y, si cayó, en qué transición?

**Ahora analizamos las etapas del funnel:** localizamos la caída y profundizamos en las transiciones prioritarias.

**Qué medimos:** conversión hacia la siguiente etapa.

**Alcance mostrado ahora:** cuentas que entraron por New; abril como referencia y mayo–agosto de 2026. Primero global; después por canal, solo en las dos transiciones con mayor caída de abril a mayo.

**Métrica:** `cuentas que avanzaron a la siguiente etapa en 30 días / cuentas con 30 días completos de seguimiento × 100`.

**Datos:** cuentas, registros de etapas y catálogo de etapas; cálculo SQL en la base del escenario, filtrado a entrada New.

**Plazo:** 30 días es una ventana inicial; la mediana New → Won no valida ese plazo para cada transición. Una caída en esta ventana puede reflejar demora.


In [ ]:
from scripts.cro_stage_focus import analyze_stage_focus
foco_etapas=analyze_stage_focus(METRICAS,lectura_new["evidencia"]["mes"])
foco_etapas["global_figure"].show(renderer="plotly_mimetype",config=CONFIG)


Conversión por etapa

#### Conclusión global


In [ ]:
display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;">'+escape(foco_etapas["global_reading"]).replace("\n", "<br>")+'</div>'))


Conclusión por etapa

#### Profundizamos en las transiciones prioritarias


In [ ]:
for detalle in foco_etapas["focused"]:
    detalle["figure"].show(renderer="plotly_mimetype",config=CONFIG)
    display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;"><strong>Conclusión:</strong> '+escape(detalle["reading"])+'</div>'))


Conversión por etapa

Conclusión por etapa

Conversión por etapa

Conclusión por etapa

### 3.5. ¿Cambió la rapidez o cobertura de atención y seguimiento?

**Qué medimos:** cuánto tardó el primer intento de contacto y qué porcentaje de visitas recibió alguno.

**Alcance mostrado ahora:** las transiciones priorizadas en 3.4, solo cuentas que entraron por New. Abril como referencia y mayo–agosto de 2026; global y por canal.

**Métricas:** `mediana(primer intento − entrada a la etapa)` en días; `visitas con intento / visitas × 100`. Los tiempos incluyen solo visitas con intento registrado.

**Datos:** cuentas, registros de etapas e interacciones. La población es la misma del foco; la unidad de atención es la visita a la etapa.


In [ ]:
from scripts.cro_focused_diagnosis import focused_diagnosis
diagnostico_focos=focused_diagnosis(METRICAS,foco_etapas,lectura_new["evidencia"]["mes"])
for bloque in diagnostico_focos["attention"]:
    bloque["figure"].show(renderer="plotly_mimetype",config=CONFIG)
    display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;"><strong>Conclusión:</strong> '+escape(bloque["reading"])+'</div>'))


Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

### 3.6. ¿Se acumulan casos lentos o todavía no tuvieron tiempo de avanzar?

**Qué medimos:** duración de las estancias terminadas y casos que siguen abiertos al corte.

**Alcance mostrado ahora:** las mismas transiciones, cuentas New y meses del foco; global y por canal. Corte: 30 de septiembre de 2026.

**Métricas:** mediana y P90 de `salida − entrada` en días; P90 deja por debajo al 90% de las estancias terminadas. Conteo de abiertos y mediana de `corte − entrada` cuando existen.

**Datos:** cuentas y registros de etapas. Avances y pérdidas cuentan como estancias terminadas; no se mezclan con los tiempos de los abiertos.


In [ ]:
for bloque in diagnostico_focos["duration"]:
    bloque["figure"].show(renderer="plotly_mimetype",config=CONFIG)
    display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;"><strong>Conclusión:</strong> '+escape(bloque["reading"])+'</div>'))


Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

### 3.7. ¿Se convierten menos o tardan más?

**Qué medimos:** si dar más tiempo recupera el avance en las transiciones priorizadas.

**Alcance mostrado ahora:** cuentas New en los mismos focos; meses comunes con 90 días completos de seguimiento (abril–junio de 2026 en referencia).

**Métrica:** `cuentas que avanzan a la siguiente etapa en 30, 60 o 90 días / entradas con seguimiento completo × 100`. Cada plazo compara las mismas cuentas y meses.

**Datos:** registros de etapas y catálogo, filtrados a entrada New.

La pregunta original sobre el mes de origen de los cierres y su análisis se conservan en el respaldo; no son necesarios para este contraste por etapa.


In [ ]:
for bloque in diagnostico_focos["windows"]:
    bloque["figure"].show(renderer="plotly_mimetype",config=CONFIG)
    display(HTML('<div style="font-family:Arial,sans-serif;font-size:19px;line-height:1.6;padding:18px 22px;border-left:4px solid #2563eb;background:#f1f5f9;color:#1e293b;border-radius:6px;"><strong>Conclusión:</strong> '+escape(bloque["reading"])+'</div>'))


Diagnóstico de transición priorizada

Conclusión del diagnóstico

Diagnóstico de transición priorizada

Conclusión del diagnóstico

## 4. Hallazgos y acciones

Acciones propuestas para los focos encontrados. Los datos permiten localizar cambios y contrastar explicaciones; todavía no acreditan sus causas.


In [ ]:
filas=[]
for hallazgo in diagnostico_focos["findings"]:
    filas.append({"Hallazgo":hallazgo["hallazgo"],"Acción propuesta":hallazgo["accion"],"Responsable":hallazgo["responsable"],"Evaluación":hallazgo["seguimiento"]})
display(pd.DataFrame(filas))


Hallazgo,Acción propuesta,Responsable,Evaluación
"Proposal → Won: conversión 94,4% → 81,4%; primer intento 2,5 → 2,0 días.",Revisar motivos de pérdida y decisiones comerciales de esta transición antes de elegir una intervención.,Sales (propuesto),Comparar atención y conversión de nuevas entradas a esta etapa con igual seguimiento; acordar plazo y criterios de éxito.
"Demo → Proposal: conversión 86,4% → 77,1%; primer intento 1,5 → 2,0 días.",Revisar las visitas sin intento y el retraso de seguimiento; probar una mejora acotada de atención.,Sales (propuesto),Comparar atención y conversión de nuevas entradas a esta etapa con igual seguimiento; acordar plazo y criterios de éxito.


Las puntuaciones aleatorias no acreditan calidad real. Si la empresa carece de esa medición, proponemos implementarla. Antes de ejecutar acciones, validar registros y definiciones contra el CRM y acordar criterios de éxito. No se han ejecutado intervenciones ni medido impacto.

## 5. Respaldo

La versión completa anterior, sus preguntas originales, gráficas y análisis complementarios permanecen en [el notebook de respaldo](archivo/02_modelo_cro_respaldo.ipynb). El recorrido principal termina en los focos y acciones anteriores.
